Unless explicitly stated otherwise all files in this repository are licensed under the Apache-2.0 License.

This product includes software developed at Datadog (https://www.datadoghq.com/)
Copyright 2026 Datadog, Inc.

# Toto 2.0 GluonTS Integration

Use `Toto2GluonTSModel` to plug Toto 2.0 into the GluonTS evaluation
pipeline. This handles data transforms, instance splitting, batched
inference, and forecast formatting automatically.

## Installation

```bash
pip install "toto-2 @ git+https://github.com/DataDog/toto.git#subdirectory=toto2"
```

In [ ]:
import torch
from toto2 import Toto2Model, Toto2GluonTSModel, Toto2GluonTSModelConfig

## Create a GluonTS predictor

Wrap the base model with `Toto2GluonTSModel` and call `create_predictor()`
to get a standard GluonTS `PyTorchPredictor`.

In [ ]:
SIZE = "4m"  # 4m | 22m | 313m | 1B | 2.5B
CHECKPOINT = f"Datadog/Toto-2.0-{SIZE}"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = Toto2Model.from_pretrained(CHECKPOINT, map_location=device)
model = model.to(device).eval()

gts_config = Toto2GluonTSModelConfig(
    prediction_length=96,
    context_length=512,
    target_dim=1,
)

gts_model = Toto2GluonTSModel(model, gts_config).to(device).eval()
predictor = gts_model.create_predictor(batch_size=64, device=device)

## Predict on a built-in dataset

GluonTS ships with many datasets. We'll use `airpassengers` (univariate, monthly).

In [ ]:
from gluonts.dataset.repository import get_dataset

# Load a built-in univariate dataset
dataset = get_dataset("airpassengers")
print(f"Frequency: {dataset.metadata.freq}")
print(f"Prediction length: {dataset.metadata.prediction_length}")

In [ ]:
from gluonts.dataset.split import split

prediction_length = dataset.metadata.prediction_length
_, test_template = split(dataset.test, offset=-prediction_length)
test_data = test_template.generate_instances(prediction_length=prediction_length, windows=1)

# Recreate predictor with this dataset's prediction length
gts_config = Toto2GluonTSModelConfig(
    prediction_length=prediction_length,
    context_length=512,
    target_dim=1,
)
gts_model = Toto2GluonTSModel(model, gts_config).to(device).eval()
predictor = gts_model.create_predictor(batch_size=64, device=device)

forecasts = list(predictor.predict(test_data.input))
print(f"Generated {len(forecasts)} forecast(s)")
print(f"Forecast type: {type(forecasts[0]).__name__}")
print(f"Quantile levels: {forecasts[0].forecast_keys}")

In [ ]:
import matplotlib.pyplot as plt

# Get the full series for plotting
entry = list(dataset.test)[0]
full_target = entry["target"]

fc = forecasts[0]
fig, ax = plt.subplots(figsize=(12, 4))

# Plot context (last 4x prediction_length)
ctx_len = min(4 * prediction_length, len(full_target) - prediction_length)
ctx = full_target[-(ctx_len + prediction_length):-prediction_length]
ax.plot(range(len(ctx)), ctx, color="black", label="Context")

# Plot ground truth
label = full_target[-prediction_length:]
ax.plot(range(len(ctx), len(ctx) + prediction_length), label, color="gray", ls="--", label="Ground truth")

# Plot forecast
x = range(len(ctx), len(ctx) + prediction_length)
ax.plot(x, fc.quantile(0.5), color="tab:blue", label="Median")
ax.fill_between(x, fc.quantile(0.1), fc.quantile(0.9), alpha=0.2, color="tab:blue", label="80% interval")

ax.legend()
ax.set_title("Toto 2.0 — Air Passengers Forecast")
plt.tight_layout()
plt.show()

## Evaluate with GluonTS metrics

In [ ]:
from gluonts.model import evaluate_model
from gluonts.ev.metrics import MAE, MASE, MSE, RMSE, SMAPE, MeanWeightedSumQuantileLoss
from gluonts.time_feature import get_seasonality

metrics = [
    MSE(forecast_type="mean"),
    MAE(),
    MASE(),
    SMAPE(),
    RMSE(forecast_type="mean"),
    MeanWeightedSumQuantileLoss(
        quantile_levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    ),
]

results = evaluate_model(
    predictor,
    test_data=test_data,
    metrics=metrics,
    axis=None,
    mask_invalid_label=True,
    allow_nan_forecast=False,
    seasonality=get_seasonality(dataset.metadata.freq),
)

for key, val in results.items():
    print(f"{key}: {val.iloc[0]:.4f}")

## Multiple series

The `electricity` dataset has 321 hourly electricity consumption series.
We forecast each one separately.

In [ ]:
# Load a built-in dataset with many series (321 hourly electricity consumption)
mv_dataset = get_dataset("electricity")
n_series = len(list(mv_dataset.train))
mv_pred_len = mv_dataset.metadata.prediction_length
print(f"electricity: {n_series} series, freq={mv_dataset.metadata.freq}, prediction_length={mv_pred_len}")

mv_config = Toto2GluonTSModelConfig(
    prediction_length=mv_pred_len,
    context_length=512,
    target_dim=1,
)

mv_model = Toto2GluonTSModel(model, mv_config).to(device).eval()
mv_predictor = mv_model.create_predictor(batch_size=32, device=device)

In [ ]:
_, mv_test_template = split(mv_dataset.test, offset=-mv_pred_len)
mv_test_data = mv_test_template.generate_instances(prediction_length=mv_pred_len, windows=1)

mv_forecasts = list(mv_predictor.predict(mv_test_data.input))
print(f"Generated {len(mv_forecasts)} forecast(s)")

# Plot first 4 series
test_entries = list(mv_dataset.test)
n_plot = min(4, len(mv_forecasts))
fig, axes = plt.subplots(1, n_plot, figsize=(14, 3))
for i, (fc, ax) in enumerate(zip(mv_forecasts[:n_plot], axes)):
    target = test_entries[i]["target"]
    ctx = target[-(4 * mv_pred_len + mv_pred_len):-mv_pred_len]
    label = target[-mv_pred_len:]
    ax.plot(range(len(ctx)), ctx, color="black")
    ax.plot(range(len(ctx), len(ctx) + mv_pred_len), label, color="gray", ls="--")
    x = range(len(ctx), len(ctx) + mv_pred_len)
    ax.plot(x, fc.quantile(0.5), color="tab:blue")
    ax.fill_between(x, fc.quantile(0.1), fc.quantile(0.9), alpha=0.2, color="tab:blue")
    ax.set_title(f"Series {i}")
fig.suptitle("Electricity Forecasts")
plt.tight_layout()
plt.show()